# Homework 10: Regression Models

**Note.** This homework has un-numbered exercises intended for further exploration (not graded). Numbered exercises are collected in Canvas as usual.

## 10.0 Introduction

This homework accompanies Lecture 10 on regression models. We apply the least-squares framework — and then the scikit-learn API — to two real datasets drawn from medicine and social science.

In Part 1 we work with the **diabetes dataset**, a classic medical benchmark. Given ten pre-standardized physiological measurements for 442 patients, we build a design matrix, solve the normal equations by hand, and measure fit quality — all without scikit-learn.

**Sections:**
- **10.1** Design matrix and the normal equations — building $\mathbf{X}$, computing $\hat{\boldsymbol{\beta}}$, and interpreting coefficients
- **10.2** Model metrics — SSR, RMSE, and $R^2$ from scratch
- **10.3** Model comparison via forward selection — finding the best model using feature selection
- **10.4** *(Optional)* Cross-validation — an honest estimate of out-of-sample error

**How to work through this notebook.** Run every cell in order before moving to the next — later cells depend on variables defined in earlier ones. Cells marked `# Run this cell without modification` are scaffolding; read them carefully but do not change them. Cells marked `# Your code here` are yours to complete.


In [ ]:
# Run this cell without modification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn import metrics

## 10.1 Design Matrix and the Normal Equations

The diabetes dataset records ten baseline physiological measurements for 442 patients, along with a quantitative measure of **disease progression** one year later. The ten features — age, sex, BMI, blood pressure, and six blood serum measurements — have already been standardized: each column has mean zero and the same sum of squares. The response $y$ is in its original units (a continuous score from 25 to 346).

Because the features are pre-standardized, a coefficient of $+500$ on `bmi` means "500 units of disease progression per one-standard-deviation increase in BMI" — not per one kg/m² increase. Keep this in mind when interpreting the coefficients below.

| Variable | Description |
|:--------:|:------------|
| `age` | Age (standardized) |
| `sex` | Biological sex (standardized) |
| `bmi` | Body mass index (standardized) |
| `bp` | Average blood pressure (standardized) |
| `s1`–`s6` | Blood serum measurements (standardized) |
| `y` | Disease progression score one year later (response) |


In [ ]:
# Run this cell without modification
data = load_diabetes()

X_raw         = data.data          # shape (442, 10) — already standardized
y             = data.target        # shape (442,)  — disease progression score
feature_names = list(data.feature_names)

print(f'X_raw shape: {X_raw.shape}')
print(f'y range:     {y.min():.1f}  to  {y.max():.1f}')
print(f'y mean:      {y.mean():.4f}')
print()
print('First three rows of X_raw:')
print(np.round(X_raw[:3], 4))

In [ ]:
# Your code here
n = ???
X = ???

print(f'X shape: {X.shape}')
print()
print('First three rows of X (intercept column, then features):')
print(np.round(X[:3], 4))

In [ ]:
# Your code here
beta_hat = ???

print('Estimated coefficients beta_hat:')
labels = ['intercept'] + feature_names
for name, val in zip(labels, beta_hat.flatten()):
    print(f'  {name:<12}: {val:+.4f}')

**Exercise 1.** The intercept $\hat{\beta}_0 = $ \_\_\_\_.

**Exercise 2.** In the data the values of "female" and "male" are encoded as 0 and 1 respectively. What predicted amount of $y$ does the model attribute to the difference between female and male?


## 10.2 Model Metrics: SSR, RMSE, and $R^2$

With $\hat{\boldsymbol{\beta}}$ in hand we can evaluate how well the model fits the data. Recall the three standard metrics:

$$\text{SSR} = \|\boldsymbol{\epsilon}\|^2, \qquad \text{RMSE} = \sqrt{\frac{\text{SSR}}{n}}, \qquad R^2 = \frac{\text{TSS} - \text{SSR}}{\text{TSS}}, \quad \text{where} \quad \text{TSS} = \sum_{i=1}^n (y_i - \bar{y})^2$$

RMSE is in the same units as $y$. $R^2$ is the fraction of the total variation in $y$ that the model explains; $R^2 = 1$ is a perfect fit and $R^2 = 0$ means the model does no better than always predicting $\bar{y}$.


In [ ]:
# Your code here
y_hat   = ???
epsilon = ???

SSR  = ???
RMSE = ???
TSS  = ???
R2   = ???

print(f'SSR  = {np.round(SSR,  4)}')
print(f'RMSE = {np.round(RMSE, 4)}  (disease progression units)')
print(f'TSS  = {np.round(TSS,  4)}')
print(f'R2   = {np.round(R2,   4)}')

**Exercise 3.** What is the RMSE for this model?

**Exercise 4.** What is the percentage of variation in disease progression score that the model can explain? Format your answer as a percent rounded to 2 decimal places.


In [ ]:
# Run this cell without modification
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(y, y_hat, color='steelblue', alpha=0.4, edgecolors='white', lw=0.4, s=20)
lims = [y.min() - 10, y.max() + 10]
ax.plot(lims, lims, 'k--', lw=1, alpha=0.7, label='Perfect prediction')
ax.set_xlabel('Actual disease progression score')
ax.set_ylabel('Predicted disease progression score')
ax.set_title(r'Actual vs. Predicted: $\mathbf{y}$ vs. $\hat{\mathbf{y}}$')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

ax = axes[1]
ax.hist(epsilon.flatten(), bins=30, color='tomato', edgecolor='white', lw=0.4)
ax.axvline(0, color='black', lw=1.2, linestyle='--')
ax.set_xlabel(r'Residual $\epsilon_i = y_i - \hat{y}_i$')
ax.set_ylabel('Count')
ax.set_title(r'Distribution of Residuals $\boldsymbol{\epsilon}$')
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle(f'Diabetes Regression  —  RMSE = {np.round(float(RMSE), 2)},  $R^2$ = {np.round(float(R2), 4)}', fontsize=12)
plt.tight_layout()
plt.show()

**Exercise.** The actual vs. predicted plot fans out at higher values of $y$ — the model's errors are larger for patients with high disease progression scores. What does this suggest about whether a single linear model is equally reliable across all patients?


## 10.3 Model Comparison via Forward Selection

In Sections 10.1–10.2 we fit a model using all ten features. But is the 10-feature model the *best* model? Adding more features always reduces RMSE on the data used for fitting — but that does not mean more features is always better. A simpler model is less likely to "memorize" the training data and tends to generalize better to new patients never seen before. This trade-off between fit quality and model complexity is central to machine learning.

A practical strategy for navigating this trade-off is **forward selection**: start with the single feature that gives the lowest RMSE, then greedily add the one additional feature that reduces RMSE the most, stopping when you have reached your desired model size.

The helper function below fits a least-squares model and returns its RMSE — the same computation we did in Section 10.2, packaged for reuse.


In [ ]:
# Run this cell without modification
def fit_and_rmse(feature_matrix, target_vector):
    """Fit a least-squares model and return RMSE."""
    x_hat   = np.linalg.inv(feature_matrix.T @ feature_matrix) @ feature_matrix.T @ target_vector
    y_hat_f = feature_matrix @ x_hat
    eps     = target_vector - y_hat_f
    return float(np.sqrt(np.sum(eps**2) / len(target_vector)))

The code cell below fits ten single-feature models — one for each feature in `X_raw`, each with an intercept column prepended. Use it to compute the RMSE for each and print the results.


In [ ]:
# Run this cell without modification.
print('Single-feature model RMSE:')
for i, name in enumerate(feature_names):
    Xi     = np.concatenate([np.ones((n, 1)), X_raw[:, [i]]], axis=1)
    rmse_i = fit_and_rmse(Xi, y)
    print(f'  {name:<12}: RMSE = {rmse_i:.4f}')

**Exercise 5.** If you could only use a single input variable to predict $\mathbf{y}$ which one would you choose based on the RMSE metric of model performance?


In [ ]:
# Run this cell without modification.
print('2-feature models: bmi + one additional feature')
for i, name in enumerate(feature_names):
    if name == 'bmi':
        continue
    Xi     = np.concatenate([np.ones((n, 1)), X_raw[:, [2, i]]], axis=1)
    rmse_i = fit_and_rmse(Xi, y)
    print(f'  bmi + {name:<12}: RMSE = {rmse_i:.4f}')

**Exercise 6.** Now that you have chosen a single best variable to use for prediction, you're considering adding a second input. Which one would you include, based on RMSE?


In [ ]:
# Run this cell without modification
print('3-feature models: bmi + s5 + one additional feature')
for i, name in enumerate(feature_names):
    if name in ('bmi', 's5'):
        continue
    Xi     = np.concatenate([np.ones((n, 1)), X_raw[:, [2, 8, i]]], axis=1)
    rmse_i = fit_and_rmse(Xi, y)
    print(f'  bmi + s5 + {name:<12}: RMSE = {rmse_i:.4f}')

In [ ]:
# Run this cell without modification
selected   = [2, 8]          # bmi, s5 — best 1- and 2-feature choices from above
remaining  = [i for i in range(len(feature_names)) if i not in selected]
rmse_curve = []

# Compute RMSE for models of size 1 through 10 via greedy forward selection
for step in range(1, len(feature_names) + 1):
    if step <= len(selected):
        cols = selected[:step]
    else:
        # Find the next best feature to add
        best_rmse, best_idx = np.inf, None
        for i in remaining:
            Xi     = np.concatenate([np.ones((n, 1)), X_raw[:, selected + [i]]], axis=1)
            rmse_i = fit_and_rmse(Xi, y)
            if rmse_i < best_rmse:
                best_rmse, best_idx = rmse_i, i
        selected.append(best_idx)
        remaining.remove(best_idx)
        cols = selected

    Xi = np.concatenate([np.ones((n, 1)), X_raw[:, cols]], axis=1)
    rmse_curve.append(fit_and_rmse(Xi, y))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(1, len(feature_names) + 1), rmse_curve,
        color='steelblue', lw=2, marker='o', markersize=7)
for step, (r, idx) in enumerate(zip(rmse_curve, selected), start=1):
    ax.annotate(feature_names[idx], xy=(step, r),
                xytext=(5, 6), textcoords='offset points', fontsize=8)
ax.set_xlabel('Number of features')
ax.set_ylabel('RMSE (disease progression units)')
ax.set_title('Forward Selection Elbow Plot')
ax.set_xticks(range(1, len(feature_names) + 1))
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**Exercise 7.** A simpler model is better because it is more likely to *generalize*: perform well when making predictions on novel data. The above elbow plot indicates that you should go with a model that has either \_\_\_ or \_\_\_ features. Hint: Where does it "elbow?"


## 10.4 Optional: Cross-Validation

### The Problem with Training RMSE

In Section 10.4 we used RMSE to decide how many features to include. But there is a subtle problem: every RMSE we computed was measured on the **same 442 patients we used to fit the model**. This is called the **training RMSE**, and it is optimistic by construction — the least-squares solution $\hat{\boldsymbol{\beta}}$ was chosen specifically to minimize error on these patients. Adding any feature, even a useless one, can only decrease training RMSE or leave it unchanged. It can never go up. This means training RMSE will always point toward the largest model, and the elbow plot from Section 10.4 will flatten but never turn upward to warn you that you have gone too far.

What we actually care about is how well the model predicts **new patients it has never seen**. A model that has memorized the quirks of the 442 training patients — rather than learning a genuine relationship between features and disease progression — will perform much worse on new data. This gap between training performance and real-world performance is called **overfitting**.

### The Idea of Cross-Validation

**$k$-fold cross-validation** gives us an honest estimate of out-of-sample RMSE without requiring a separate dataset. The procedure is:

1. Randomly divide the $n$ patients into $k$ equally sized groups called **folds**.
2. For each fold $j = 1, \ldots, k$:
   - **Train** the model on the $k - 1$ folds that are *not* fold $j$ (the training set).
   - **Evaluate** the model on fold $j$ (the validation set) — data the model never saw during training.
   - Record the RMSE on fold $j$.
3. Average the $k$ validation RMSEs. This average is the **cross-validated RMSE** (CV-RMSE).

Because each patient is evaluated exactly once as part of a held-out fold, CV-RMSE measures genuine predictive accuracy on unseen data. Unlike training RMSE, it **will** increase if you add features that do not generalize — producing a true elbow in the curve.

With $k = 5$ folds and $n = 442$ patients, each training set has $442 \times \frac{4}{5} \approx 354$ patients and each validation set has $\approx 88$ patients. This is illustrated below:

$$\underbrace{\overbrace{\text{fold 1}}^{\text{validate}} \mid \text{fold 2} \mid \text{fold 3} \mid \text{fold 4} \mid \text{fold 5}}_{\text{full dataset}}$$
$$\text{fold 1} \mid \overbrace{\text{fold 2}}^{\text{validate}} \mid \text{fold 3} \mid \text{fold 4} \mid \text{fold 5}$$
$$\vdots$$

Each row is one pass. At the end, every patient has been in the validation set exactly once.

### Why Not Just Use a Hold-Out Test Set?

You could instead split the data once into a training set and a test set, fit on the training portion, and evaluate on the test portion. This works, but it wastes data: a model fit on 70% of 442 patients is fit less well than one trained on all 442. Cross-validation gets the best of both worlds — every patient is used for training $k - 1$ times and for validation once — so the CV-RMSE estimate uses the full dataset without any data leakage.


In [ ]:
# Run this cell without modification
from sklearn.model_selection import KFold

def cv_rmse(col_indices, X, y_vec, k=5, seed=0):
    """Return mean RMSE across k validation folds for a model using col_indices features."""
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    fold_rmses = []
    for train_idx, val_idx in kf.split(X):
        X_tr = np.concatenate([np.ones((len(train_idx), 1)), X[np.ix_(train_idx, col_indices)]], axis=1)
        X_va = np.concatenate([np.ones((len(val_idx),  1)), X[np.ix_(val_idx,  col_indices)]], axis=1)
        y_tr = y_vec[train_idx]
        y_va = y_vec[val_idx]
        beta = np.linalg.inv(X_tr.T @ X_tr) @ X_tr.T @ y_tr
        eps  = y_va - X_va @ beta
        fold_rmses.append(float(np.sqrt(np.sum(eps**2) / len(val_idx))))
    return float(np.mean(fold_rmses))

In [ ]:
# Run this cell without modification
selected_cv  = []
remaining_cv = list(range(len(feature_names)))
cv_curve     = []
train_curve  = []

for step in range(len(feature_names)):
    # Greedy: pick the feature that minimises CV-RMSE when added next
    best_cv_rmse, best_idx = np.inf, None
    for i in remaining_cv:
        r = cv_rmse(selected_cv + [i], X_raw, y)
        if r < best_cv_rmse:
            best_cv_rmse, best_idx = r, i
    selected_cv.append(best_idx)
    remaining_cv.remove(best_idx)
    cv_curve.append(best_cv_rmse)

    # Training RMSE on the same feature set (for comparison)
    Xi = np.concatenate([np.ones((n, 1)), X_raw[:, selected_cv]], axis=1)
    train_curve.append(fit_and_rmse(Xi, y))

steps = range(1, len(feature_names) + 1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(steps, train_curve, color='steelblue', lw=2, marker='o', markersize=7, label='Training RMSE')
ax.plot(steps, cv_curve,    color='tomato',    lw=2, marker='o', markersize=7, label='CV-RMSE (5-fold)')
for step, (cv, idx) in enumerate(zip(cv_curve, selected_cv), start=1):
    ax.annotate(feature_names[idx], xy=(step, cv),
                xytext=(5, 6), textcoords='offset points', fontsize=8)
best_step = int(np.argmin(cv_curve)) + 1
ax.axvline(best_step, color='tomato', lw=1, linestyle='--', alpha=0.6,
           label=f'CV minimum  ({best_step} features)')
ax.set_xlabel('Number of features')
ax.set_ylabel('RMSE (disease progression units)')
ax.set_title('Forward Selection: Training RMSE vs. CV-RMSE')
ax.set_xticks(list(steps))
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print('CV selection order:')
for step, idx in enumerate(selected_cv, start=1):
    print(f'  Step {step}: {feature_names[idx]:<12}  CV-RMSE = {cv_curve[step-1]:.4f}  '
          f'Train-RMSE = {train_curve[step-1]:.4f}')
print(f'\nCV-optimal model size: {best_step} features')

**Exercise.** Training RMSE decreases at every step, but CV-RMSE eventually turns upward. Explain in your own words why these two curves behave differently as the number of features grows.

**Exercise.** The CV-optimal model has fewer features than the full 10-feature model, yet its CV-RMSE is lower than the full model's CV-RMSE. How is it possible for a *simpler* model to make *better* predictions on new data?

**Exercise.** We used $k = 5$ folds. What would happen to the CV-RMSE estimate and its variability if you increased $k$ to 10 or all the way to $k = n$ (leave-one-out cross-validation)? What would be the computational cost?
